# Deep Learning 基礎講座　最終課題: 脳波分類

## 概要
被験者が画像を見ているときの脳波から，その画像がどのカテゴリに属するかを分類するタスク．
- サンプル数: 訓練 118,800 サンプル，検証 59,400 サンプル，テスト 59,400 サンプル
- クラス数: 5
- 入力: 脳波データ（チャンネル数 x 系列長）
- 出力: 対応する画像のクラス
- 評価指標: Top-1 accuracy

### 元データセット ([Gifford2022 EEG dataset](https://osf.io/3jk45/)) との違い

- 本コンペでは難易度調整の目的で元データセットにいくつかの改変を加えています．

1. 訓練セットのみの使用
  - 元データセットでは訓練データに存在しなかったクラスの画像を見ているときの脳波においてテストが行われますが，これは難易度が非常に高くなります．
  - 本コンペでは元データセットの訓練セットを再分割し，訓練時に存在した画像に対応する別の脳波において検証・テストを行います．

2. クラス数の減少
  - 元データセット（の訓練セット）では16,540枚の画像に対し，1,654のクラスが存在します．
    - e.g. `aardvark`, `alligator`, `almond`, ...
  - 本コンペでは1,654のクラスを，`animal`, `food`, `clothing`, `tool`, `vehicle`の5つにまとめています．
    - e.g. `aardvark -> animal`, `alligator -> animal`, `almond -> food`, ...

### 考えられる工夫の例

- 音声モデルの導入
  - 脳波と同じ波である音声を扱うアーキテクチャを用いることが有効であると知られています．
  - 例）Conformer [[Gulati+ 2020](https://arxiv.org/abs/2005.08100)]
- 画像データを用いた事前学習
  - 本コンペのタスクは脳波のクラス分類ですが，配布してある画像データを脳波エンコーダの事前学習に用いることを許可します．
  - 例）CLIP [Radford+ 2021]
  - 画像を用いる場合は[こちら](https://osf.io/download/3v527/)からダウンロードしてください．
- 過学習を防ぐ正則化やドロップアウト


## 修了要件を満たす条件
- ベースラインモデルのbest test accuracyは38.7%となります．**これを超えた提出のみ，修了要件として認めます**．
- ベースラインから改善を加えることで，55%までは性能向上することを運営で確認しています．こちらを 1 つの指標として取り組んでみてください．

## 注意点
- 学習するモデルについて制限はありませんが，必ず訓練データで学習したモデルで予測してください．
    - 事前学習済みモデルを利用して，訓練データを fine-tuning しても構いません．
    - 埋め込み抽出モデルなど，モデルの一部を訓練しないケースは構いません．
    - 学習を一切せずに，ChatGPT などの基盤モデルを利用することは禁止とします．

## 1.準備

In [1]:
# omnicampus 実行用
# !pip install ipywidgets
!pip install transformers
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 59.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 46.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 73.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.5 MB/s e

In [12]:
# ライブラリのインポートとシード固定
import os, sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from einops.layers.torch import Rearrange
from einops import repeat
from glob import glob
from termcolor import cprint
from tqdm.notebook import tqdm
# from transformers import (
#     TimeSeriesTransformerConfig,
#     TimeSeriesTransformerModel,
#     TimeSeriesTransformerForPrediction
# )
import math
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [13]:
# ドライブのマウント（Colabの場合）
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# ワーキングディレクトリを作成し移動．ノートブックを配置したディレクトリに適宜書き換え
# WORK_DIR = "/workspace/assets"
WORK_DIR = '/content/drive/MyDrive/DLBasic/final_project'
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

/content/drive/MyDrive/DLBasic/final_project


## 2.データセット

ノートブックと同じディレクトリに`data/`が存在することを確認してください．

In [15]:
class EEGAugmentation:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, x):
        if np.random.random() < self.p:
            # Time shifting
            shift = np.random.randint(-5, 6)
            x = torch.roll(x, shifts=shift, dims=-1)

            # Channel dropout
            if np.random.random() < 0.3:
                num_channels = x.shape[0]
                drop_channels = np.random.choice(num_channels, size=max(1, num_channels//8), replace=False)
                x[drop_channels] = 0

            # Gaussian noise
            if np.random.random() < 0.4:
                noise = torch.randn_like(x) * 0.02
                x = x + noise

            # Amplitude scaling
            if np.random.random() < 0.3:
                scale = np.random.uniform(0.8, 1.2)
                x = x * scale

        return x

In [16]:
class ThingsEEGDataset(torch.utils.data.Dataset):
    def __init__(self, split: str, apply_whitening: bool = False, normalize_per_subject: bool = True, augment: bool = False) -> None:
        super().__init__()
        assert split in ["train", "val", "test"], f"Invalid split: {split}"
        self.split = split
        self.num_classes = 5
        self.num_subjects = 10
        self.normalize_per_subject = normalize_per_subject
        self.augment = augment and split == "train"

        self.X = np.load(f"data/{split}/eeg.npy")
        self.X = torch.from_numpy(self.X).to(torch.float32)
        self.subject_idxs = np.load(f"data/{split}/subject_idxs.npy")
        self.subject_idxs = torch.from_numpy(self.subject_idxs)

        if split in ["train", "val"]:
            self.y = np.load(f"data/{split}/labels.npy")
            self.y = torch.from_numpy(self.y)

        # normalization
        if self.normalize_per_subject:
            self._normalize_per_subject()
        else:
            # 全体での正規化
            self.X = (self.X - self.X.mean(dim=-1, keepdim=True)) / (self.X.std(dim=-1, keepdim=True) + 1e-5)

        self.apply_whitening = apply_whitening
        if self.apply_whitening:
            self._apply_whitening()

        if self.augment:
            self.augmentation = EEGAugmentation(p=0.6)

        self.num_subjects = len(torch.unique(self.subject_idxs))
        print(f"EEG: {self.X.shape}, labels: {self.y.shape if hasattr(self, 'y') else None}, subject indices: {self.subject_idxs.shape}")
        print(f"Number of subjects: {self.num_subjects}")

    def _normalize_per_subject(self):
        """被験者ごとの正規化"""
        for subject_id in torch.unique(self.subject_idxs):
            mask = self.subject_idxs == subject_id
            subject_data = self.X[mask]
            # チャネルごとに正規化
            mean = subject_data.mean(dim=(0, 2), keepdim=True)
            std = subject_data.std(dim=(0, 2), keepdim=True)
            self.X[mask] = (subject_data - mean) / (std + 1e-5)

    def _apply_whitening(self):
        """ホワイトニング処理"""
        X_flat = self.X.reshape(self.X.shape[0], -1).numpy()
        X_mean = X_flat.mean(axis=0, keepdims=True)
        X_centered = X_flat - X_mean
        cov = np.cov(X_centered, rowvar=False)

        U, S, Vt = np.linalg.svd(cov)

        S_clipped = np.maximum(S, 1e-3)
        self.whitening_matrix = torch.from_numpy(U @ np.diag(1.0 / np.sqrt(S_clipped)) @ U.T).to(torch.float32)
        self.X_mean = torch.from_numpy(X_mean).to(torch.float32)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, i):
        X = self.X[i]
        if self.apply_whitening:
            X_flat = X.flatten()
            X_flat = X_flat - self.X_mean.squeeze()
            X_flat = torch.matmul(self.whitening_matrix, X_flat)
            X = X_flat.view_as(X)

        if self.augment:
            X = self.augmentation(X)

        if hasattr(self, "y"):
            return X, self.y[i], self.subject_idxs[i]
        else:
            return X, self.subject_idxs[i]

    @property
    def num_channels(self) -> int:
        return self.X.shape[1]

    @property
    def seq_len(self) -> int:
        return self.X.shape[2]

## 3.ベースラインモデル

In [17]:
# Conformer-inspired architecture for EEG data
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        output = torch.matmul(attention_weights, V)
        return output

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        attention = self.scaled_dot_product_attention(Q, K, V, mask)
        attention = attention.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        output = self.W_o(attention)
        return output

class ConvolutionModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, dropout=0.1):
        super().__init__()
        self.pointwise_conv1 = nn.Conv1d(d_model, d_model * 2, kernel_size=1)
        self.depthwise_conv = nn.Conv1d(d_model, d_model, kernel_size=kernel_size,
                                       padding=(kernel_size - 1) // 2, groups=d_model)
        self.batch_norm = nn.BatchNorm1d(d_model)
        self.pointwise_conv2 = nn.Conv1d(d_model, d_model, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x.transpose(1, 2)  # (batch, d_model, seq_len)

        x = self.pointwise_conv1(x)
        x = F.glu(x, dim=1)
        x = self.depthwise_conv(x)
        x = self.batch_norm(x)
        x = F.silu(x)
        x = self.pointwise_conv2(x)
        x = self.dropout(x)

        return x.transpose(1, 2)  # (batch, seq_len, d_model)

class ConformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, conv_kernel_size=31, dropout=0.1):
        super().__init__()
        self.ff1 = FeedForward(d_model, d_ff, dropout)
        self.mha = MultiHeadAttention(d_model, num_heads, dropout)
        self.conv = ConvolutionModule(d_model, conv_kernel_size, dropout)
        self.ff2 = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.norm4 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Feed forward 1
        x = x + 0.5 * self.dropout(self.ff1(self.norm1(x)))

        # Multi-head attention
        x = x + self.dropout(self.mha(self.norm2(x), self.norm2(x), self.norm2(x)))

        # Convolution
        x = x + self.dropout(self.conv(self.norm3(x)))

        # Feed forward 2
        x = x + 0.5 * self.dropout(self.ff2(self.norm4(x)))

        return x

class EEGConformerClassifier(nn.Module):
    def __init__(self, num_classes, seq_len, in_channels, d_model=256, num_heads=8,
                 num_layers=4, d_ff=1024, dropout=0.1):
        super().__init__()

        # Input projection
        self.input_projection = nn.Conv1d(in_channels, d_model, kernel_size=3, padding=1)

        # Positional encoding
        self.pos_encoding = nn.Parameter(torch.randn(1, seq_len, d_model))

        # Conformer blocks
        self.conformer_blocks = nn.ModuleList([
            ConformerBlock(d_model, num_heads, d_ff, dropout=dropout)
            for _ in range(num_layers)
        ])

        # Classification head
        self.norm = nn.LayerNorm(d_model)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x):
        # x: (batch, channels, seq_len)
        batch_size = x.size(0)

        # Input projection
        x = self.input_projection(x)  # (batch, d_model, seq_len)
        x = x.transpose(1, 2)  # (batch, seq_len, d_model)

        # Add positional encoding
        x = x + self.pos_encoding

        # Apply conformer blocks
        for block in self.conformer_blocks:
            x = block(x)

        # Normalization
        x = self.norm(x)

        # Global pooling and classification
        x = x.transpose(1, 2)  # (batch, d_model, seq_len)
        x = self.global_pool(x).squeeze(-1)  # (batch, d_model)
        x = self.classifier(x)

        return x

# Pre-training for EEG representation learning
class EEGPreTrainingModel(nn.Module):
    def __init__(self, seq_len, in_channels, d_model=256, num_heads=8,
                 num_layers=4, d_ff=1024, dropout=0.1):
        super().__init__()

        # Input projection
        self.input_projection = nn.Conv1d(in_channels, d_model, kernel_size=3, padding=1)

        # Positional encoding
        self.pos_encoding = nn.Parameter(torch.randn(1, seq_len, d_model))

        # Conformer blocks
        self.conformer_blocks = nn.ModuleList([
            ConformerBlock(d_model, num_heads, d_ff, dropout=dropout)
            for _ in range(num_layers)
        ])

        # Pre-training heads
        self.norm = nn.LayerNorm(d_model)
        self.reconstruction_head = nn.Linear(d_model, in_channels)
        self.contrastive_head = nn.Linear(d_model, 128)

    def forward(self, x, return_features=False):
        # x: (batch, channels, seq_len)
        batch_size = x.size(0)

        # Input projection
        x_proj = self.input_projection(x)  # (batch, d_model, seq_len)
        x_proj = x_proj.transpose(1, 2)  # (batch, seq_len, d_model)

        # Add positional encoding
        x_proj = x_proj + self.pos_encoding

        # Apply conformer blocks
        features = x_proj
        for block in self.conformer_blocks:
            features = block(features)

        # Normalization
        features = self.norm(features)

        if return_features:
            return features

        # Reconstruction
        reconstruction = self.reconstruction_head(features)  # (batch, seq_len, in_channels)
        reconstruction = reconstruction.transpose(1, 2)  # (batch, in_channels, seq_len)

        # Contrastive features
        contrastive = self.contrastive_head(features.mean(dim=1))  # (batch, 128)

        return reconstruction, contrastive

## 4.訓練実行

In [18]:
def pretrain_model(model, train_loader, val_loader, epochs=20, lr=1e-4):
    """Pre-train the model using reconstruction and contrastive learning"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    model.train()
    best_loss = float('inf')

    for epoch in range(epochs):
        total_loss = 0
        total_recon_loss = 0
        total_contrast_loss = 0

        for batch_idx, (X, y, subject_idxs) in enumerate(tqdm(train_loader, desc=f"Pre-training Epoch {epoch+1}")):
            X = X.to("cuda")

            # Add noise for denoising autoencoder
            noise = torch.randn_like(X) * 0.1
            X_noisy = X + noise

            # Forward pass
            reconstruction, contrastive = model(X_noisy)

            # Reconstruction loss
            recon_loss = F.mse_loss(reconstruction, X)

            # Contrastive loss (InfoNCE-like)
            batch_size = contrastive.size(0)
            if batch_size > 1:
                # Positive pairs: same subject
                labels = subject_idxs.to("cuda")
                similarity_matrix = torch.mm(contrastive, contrastive.t())

                # Create mask for positive pairs
                pos_mask = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
                pos_mask.fill_diagonal_(0)  # Remove self-similarity

                # Compute contrastive loss
                if pos_mask.sum() > 0:
                    pos_sim = similarity_matrix * pos_mask
                    neg_sim = similarity_matrix * (1 - pos_mask)

                    contrast_loss = -torch.log(
                        torch.exp(pos_sim).sum(dim=1) /
                        (torch.exp(pos_sim).sum(dim=1) + torch.exp(neg_sim).sum(dim=1) + 1e-8)
                    ).mean()
                else:
                    contrast_loss = torch.tensor(0.0, device=X.device)
            else:
                contrast_loss = torch.tensor(0.0, device=X.device)

            # Total loss
            loss = recon_loss + 0.1 * contrast_loss

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_recon_loss += recon_loss.item()
            total_contrast_loss += contrast_loss.item()

        scheduler.step()

        avg_loss = total_loss / len(train_loader)
        avg_recon_loss = total_recon_loss / len(train_loader)
        avg_contrast_loss = total_contrast_loss / len(train_loader)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Loss: {avg_loss:.4f} | "
              f"Recon: {avg_recon_loss:.4f} | "
              f"Contrast: {avg_contrast_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            cprint("New pretrained model saved. ")
            torch.save(model.state_dict(), "pretrained_model.pt")

    return model

In [19]:
def create_finetuning_model(pretrained_model, num_classes, freeze_encoder=False):
    """Create fine-tuning model from pre-trained model"""
    # Extract encoder components
    finetuning_model = EEGConformerClassifier(
        num_classes=num_classes,
        seq_len=100,  # Based on the data shape
        in_channels=17,
        d_model=256,
        num_heads=8,
        num_layers=4,
        d_ff=1024,
        dropout=0.1
    )

    # Load pre-trained weights
    pretrained_state = pretrained_model.state_dict()
    finetuning_state = finetuning_model.state_dict()

    # Transfer weights for encoder components
    for name in finetuning_state.keys():
        if name in pretrained_state:
            finetuning_state[name] = pretrained_state[name]

    finetuning_model.load_state_dict(finetuning_state)

    # Freeze encoder if requested
    if freeze_encoder:
        for name, param in finetuning_model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad = False

    return finetuning_model

In [20]:
def train_with_pretraining(pretrained=False, start_epoch=0):
    # Hyperparameters
    batch_size = 128  # Smaller batch size for larger model
    pretrain_epochs = 25
    finetune_epochs = 80
    pretrain_lr = 1e-4
    finetune_lr = 3e-5

    # Data loading
    train_set = ThingsEEGDataset("train", augment=True)
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=batch_size, shuffle=True
    )
    val_set = ThingsEEGDataset("val", augment=False)
    val_loader = torch.utils.data.DataLoader(
        val_set, batch_size=batch_size, shuffle=False
    )

    print(f"Number of subjects: {train_set.num_subjects}")

    if not pretrained:

        # Step 1: Pre-training
        print("=" * 50)
        print("Step 1: Pre-training")
        print("=" * 50)

        pretrain_model_instance = EEGPreTrainingModel(
            seq_len=train_set.seq_len,
            in_channels=train_set.num_channels,
            d_model=256,
            num_heads=8,
            num_layers=4,
            d_ff=1024,
            dropout=0.1
        ).to("cuda")

        pretrain_model_instance = pretrain_model(
            pretrain_model_instance, train_loader, val_loader,
            epochs=pretrain_epochs, lr=pretrain_lr
        )

        pretrained = True

    else:
        pretrain_model_instance = EEGPreTrainingModel(
            seq_len=train_set.seq_len,
            in_channels=train_set.num_channels,
            d_model=256,
            num_heads=8,
            num_layers=4,
            d_ff=1024,
            dropout=0.1
        ).to("cuda")
        pretrain_model_instance.load_state_dict(torch.load("./conformer_0.50114/pretrained_model.pt"))

    max_val_acc = 0
    patience = 10
    patience_counter = 0
    if pretrained:

        # Step 2: Fine-tuning
        print("=" * 50)
        print("Step 2: Fine-tuning")
        print("=" * 50)

        model = create_finetuning_model(
            pretrain_model_instance,
            num_classes=train_set.num_classes,
            freeze_encoder=False  # Set to True if you want to freeze encoder
        ).to("cuda")

        # Fine-tuning optimizer with different learning rates
        optimizer = torch.optim.AdamW([
            {'params': [p for n, p in model.named_parameters() if 'classifier' not in n], 'lr': finetune_lr},
            {'params': [p for n, p in model.named_parameters() if 'classifier' in n], 'lr': finetune_lr * 5}
        ], weight_decay=0.01, eps=1e-6)

        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

        if start_epoch >= 1:
            checkpoint = torch.load("model_last.pt")
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            max_val_acc = checkpoint['max_val_acc']
            patience_counter = checkpoint['patience_counter']

    # Training loop

    # writer = SummaryWriter("tensorboard_pretrained")

    def accuracy(y_pred, y):
        return (y_pred.argmax(dim=-1) == y).float().mean()

    for epoch in range(start_epoch, finetune_epochs):
        print(f"Epoch {epoch+1}/{finetune_epochs}")

        train_loss, train_acc, val_loss, val_acc = [], [], [], []

        # Training
        model.train()
        for X, y, subject_idxs in tqdm(train_loader, desc="Train"):
            X, y = X.to("cuda"), y.to("cuda")

            y_pred = model(X)
            loss = F.cross_entropy(y_pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss.append(loss.item())
            train_acc.append(accuracy(y_pred, y).item())

        # Validation
        model.eval()
        for X, y, subject_idxs in tqdm(val_loader, desc="Validation"):
            X, y = X.to("cuda"), y.to("cuda")

            with torch.no_grad():
                y_pred = model(X)

            val_loss.append(F.cross_entropy(y_pred, y).item())
            val_acc.append(accuracy(y_pred, y).item())

        scheduler.step()

        # Logging
        train_loss_avg = np.mean(train_loss)
        train_acc_avg = np.mean(train_acc)
        val_loss_avg = np.mean(val_loss)
        val_acc_avg = np.mean(val_acc)

        print(f"Epoch {epoch+1}/{finetune_epochs} | "
              f"train loss: {train_loss_avg:.3f} | "
              f"train acc: {train_acc_avg:.3f} | "
              f"val loss: {val_loss_avg:.3f} | "
              f"val acc: {val_acc_avg:.3f}")

        # writer.add_scalar("train_loss", train_loss_avg, epoch)
        # writer.add_scalar("train_acc", train_acc_avg, epoch)
        # writer.add_scalar("val_loss", val_loss_avg, epoch)
        # writer.add_scalar("val_acc", val_acc_avg, epoch)

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'max_val_acc': max_val_acc,
            'patience_counter': patience_counter,
        }, "model_last.pt")

        if val_acc_avg > max_val_acc:
            print("New best. Saving the model.", "cyan")
            torch.save(model.state_dict(), "model_best.pt")
            max_val_acc = val_acc_avg
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered. ")
                break

    return model

In [21]:
model = train_with_pretraining(pretrained=True, start_epoch=0)

EEG: torch.Size([118800, 17, 100]), labels: torch.Size([118800]), subject indices: torch.Size([118800])
Number of subjects: 10
EEG: torch.Size([59400, 17, 100]), labels: torch.Size([59400]), subject indices: torch.Size([59400])
Number of subjects: 10
Number of subjects: 10


/tmp/ipython-input-20-2168046967.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrain_model_instance.load_state_dict(torch.load("./conformer_0.50114/pretrained_mode

Step 2: Fine-tuning
Epoch 1/80


Train:   0%|          | 0/929 [00:00<?, ?it/s]

Validation:   0%|          | 0/465 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/465 [00:00<?, ?it/s]

Epoch 56/80 | train loss: 1.084 | train acc: 0.581 | val loss: 1.345 | val acc: 0.505
Epoch 57/80


Train:   0%|          | 0/929 [00:00<?, ?it/s]

Validation:   0%|          | 0/465 [00:00<?, ?it/s]

Epoch 57/80 | train loss: 1.078 | train acc: 0.583 | val loss: 1.329 | val acc: 0.507
Early stopping triggered. 


## 5.評価

In [22]:
# ------------------
#    Dataloader
# ------------------
test_set = ThingsEEGDataset("test")
test_loader = torch.utils.data.DataLoader(
    test_set, batch_size=128, shuffle=False
)

# ------------------
#       Model
# ------------------
model = EEGConformerClassifier(
        num_classes=5,
        seq_len=100,
        in_channels=17,
        d_model=256,
        num_heads=8,
        num_layers=4,
        d_ff=1024,
        dropout=0.1
).to("cuda")

model.load_state_dict(torch.load("model_best.pt", map_location="cuda"))

# ------------------
#  Start evaluation
# ------------------
preds = []
model.eval()
for X, subject_idxs in tqdm(test_loader, desc="Evaluation"):
    preds.append(model(X.to("cuda")).detach().cpu())

preds = torch.cat(preds, dim=0).numpy()
np.save("submission", preds)
print(f"Submission {preds.shape} saved.")

EEG: torch.Size([59400, 17, 100]), labels: None, subject indices: torch.Size([59400])
Number of subjects: 10


/tmp/ipython-input-22-3797233005.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("model_best.pt", map_location="cuda"))


Evaluation:   0%|          | 0/465 [00:00<?, ?it/s]

Submission (59400, 5) saved.


## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (EEG)」から提出してください．

- `submission.npy`
- `model_last.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [23]:
from zipfile import ZipFile

model_path = "model_best.pt"
notebook_path = "DLBasics2025_competition_EEG_conformer_v2.ipynb"

with ZipFile("submission.zip", "w") as zf:
    zf.write("submission.npy")
    zf.write(model_path)
    zf.write(notebook_path)